In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt
import os

In [ ]:
import sys
sys.path.append("../src/")

In [ ]:
from drone_core import (get_poses_from_data, get_images_from_data, read_images_data_from_folder,
                        get_cameras_data, create_nerfstudio_dataset)
from visualization import visualize_cameras
from colmap_conversion import write_colmap_cameras_txt, write_colmap_images_txt, convert_data_to_colmap

## Read and visualize data with poses from meta data

In [ ]:
src_data_dir = "G:/Work/AI/NeRF/drone_dataset/raw_data/dataset_testing/picnic_bench/images/level_1"

Read all the data from the given image folder

In [ ]:
images_data, H, W = read_images_data_from_folder(src_data_dir)
print(f"Number of images found = {len(images_data)} with image resolution H = {H} and W = {W}")
print("Sample image data:")
print(json.dumps(images_data[0], indent=4))

Visualize the images poses (c2w) based on the GPS and drone IMU data

In [ ]:
poses = get_poses_from_data(images_data)
gps_vis = visualize_cameras(poses = poses, image_size=(W, H), stride=2, trajectory_color=[0, 1, 0])

Show a sample image from the data

In [ ]:
loaded_images = get_images_from_data(images_data, src_data_dir)
rgb_image = cv2.cvtColor(loaded_images[0], cv2.COLOR_BGR2RGB)
plt.imshow(rgb_image)

## Create Nerfstudio dataset from metadata poses and opencv camera calibration

In [ ]:
dst_dataset_dir = "G:/Work/AI/NeRF/drone_dataset/processed/bench_gps"
camera_calibration_file = "../data/camera_calibration.json"

Read the camera intrinsic from camera calibration file

In [ ]:
cameras_data = get_cameras_data(cal_file=camera_calibration_file)
print(f"Number of camera intrinsic found = {len(cameras_data)}")
print(json.dumps(cameras_data[0], indent=4))

In [ ]:
# Create the nerfstudio dataset
create_nerfstudio_dataset(images_data, cameras_data[0], dst_dataset_dir, src_data_dir)

## Create COLMAP near fixed initialization

This type of COLMAP initialization works by providing a colmap model from the metadata poses and the camera intrinsic calibration on top of which COLMAP could run the point_triangulator and the bundle_sdjuster

The sequence using COLMAP CLI would be as follows

```bash
colmap feature_extractor \
--database_path <db_path> \
--image_path <image_path> \
--ImageReader.single_camera 1 \
--ImageReader.camera_model OPENCV
```

```bash
colmap exhaustive_matcher \
--database_path <db path>
```

--> Inject the camera.txt and images.txt files created here

```bash
colmap point_triangulator \
--database_path <db path> \
--image_path <image path> \
--input_path <input model> \
--output_path <optimized model>
```

```bash
colmap bundle_adjuster \
--input_path <optimized model>\
--output_path <optimized model>/1
```

In [ ]:
dst_colmap_dir  = f"{dst_dataset_dir}/COLMAP/initial_data/"
dst_colmap_images_txt = f"{dst_colmap_dir}/images.txt"
dst_colmap_cameras_txt = f"{dst_colmap_dir}/cameras.txt"
colmap_data_base = f"{dst_colmap_dir}/../database.db"

In [ ]:
os.makedirs(dst_colmap_dir, exist_ok=True)
# Write the camera calibration in camera.txt
write_colmap_cameras_txt(cameras_data, dst_colmap_cameras_txt)

In [ ]:
# Convert the images data to colmap formate (specificaly the camera extrinsic)
images_data_colmap = convert_data_to_colmap(images_data)
# Write the images.txt (Note: the colmap database hase to exist for this to work 
# to be able to read the image name to image id mapping)
write_colmap_images_txt(images_data_colmap, 
                        colmap_database=colmap_data_base, 
                        images_txt=dst_colmap_images_txt)